# GoogleNet

It uses Inception module to extract features at multiple scales while using fewer parameters. Much smaller than
- AlexNet → ~9× larger
- VGG16 → ~22× larger

Inception module:
Multiple operations happens in parallel on the same input.

```TEXT
                 ┌── 1×1 Conv ──┐
                 ├── 3×3 Conv ──┤
Input Feature ───┼── 5×5 Conv ──┼──→ Concatenate
    Map           └── MaxPool ───┘

```

*Four Parallel paths*
* $1*1$ convolution
* $3*3$ convolution
* $5*5$ convolution
* Max Pooling
All operate on the same input feature map


Different kernel sizes see different amounts of images.
```
1×1 → very small/local information
3×3 → medium-scale features
5×5 → larger-scale features

inception = "look at the image in different ways at the time"
```

**Concatenation**: after four operations, their feature maps are concatenated.
```
1×1 features ─┐
3×3 features ─┤
5×5 features ─┼──→ Concatenate → Next layer
Pool features ─┘

Spatial dimension = the height × width of the output (e.g., 28×28).

the outputs must have same height and width. Therefore padding and stride are chosen so that: All four outputs have same spatial dimensions.

```

Deeper CNNS can suffer from:
- more computation
- Overfitting
- Vanishing gradients


1. $1*1$  Convolution = reduce channels before expensive convolution.
- It can also increase non-linearity because ReLU is applied after it.
```
Example
Without 1×1:
5×5×128×256 = 819,200 parameters

With 1×1:
1×1×128×64 + 5×5×64×256
= 417,792
```


2. $1*1$ after Max pooling:
```
The pooling branch can also produce many channels
so,
MaxPool
   ↓
1×1 Conv
   ↓
Reduced channels
   ↓
Concatenate


This keeps the final inception module from becoming unnecessarily large.
```

3. Average Pooling instead of many dense layers:
Others used  fully connected layers but googlenet replaces them with:
```
Feature Maps
     ↓
Average Pooling
     ↓
Classification



VGG16 → ~89% of parameters in final 3 FC layers
AlexNet → ~95% in final FC layers
```

Because fully connected layers can contain huge number of parameters.
It has fewer parameters, average pooling=reduce spatial dimension without a huge dense section.

4. Auxiliary Classifier:
GoogleNet can go very deep, can cause vanishing gradients, so we add small extra classifiers in the middle of networks.

```
                → Auxiliary Classifier
                ↓
Input → CNN → CNN → CNN → CNN → Final Classifier


It can calculate loss also :

Main Loss
   +
Auxiliary Loss
   ↓
Total Loss

This gives earlier layers a strong gradient signal.
```

Auxilary classifiers are used during training, removed during inference.


▶  Architecture:
```
Input
 ↓
Conv + Pool
 ↓
2 × Inception
 ↓
Pool
 ↓
5 × Inception
 ↓
Pool
 ↓
2 × Inception
 ↓
Average Pool
 ↓
Fully Connected
 ↓
Classification


Auxiliary classifiers branch out around:
Inception 4a
Inception 4d
```

| Term                      | Meaning                                          |
| ------------------------- | ------------------------------------------------ |
| **1×1 Conv / Bottleneck** | Reduce channels before expensive convolution     |
| **Average Pooling**       | Reduce spatial dimensions using averages         |
| **Auxiliary Classifier**  | Extra classifier used during training            |
| **Vanishing Gradient**    | Gradients become extremely small in early layers |
| **Inference**             | Using the trained model to make predictions      |




*GoogLeNet's Main Tricks*

1×1 Conv
→ Reduce computation

Inception
→ Multiple scales in parallel

Average Pooling
→ Avoid huge Dense layers

Auxiliary Classifier
→ Help gradients reach early layers

<br>

VGG = deeper with simple 3×3 blocks

GoogLeNet = parallel Inception blocks + 1×1 bottlenecks + fewer parameters

In [ ]:
import torch
import torch.nn as nn

class BaseConv2D(nn.Module):
  def __init__(self,in_channels, out_channels, **kwargs):
    super(BaseConv2d,self).__init__()
    self.conv=nn.Conv2d(in_channels,out_channels,**kwargs)
    self.ReLU()

  def forward(self,x):
    x=self.conv(x)
    y=self.relu(x)
    return x

class InceptionModule(nn.Module):
  def __init__(self,in_channels,n1x1,n3x3red,n3x3,n5x5red,n5x5,pool_proj):
    super(InceptionModule,self).__init__()

    self.b1=nn.Sequential(
        nn.Conv2d(in_channels,n1x1,kernel_size=1),
        nn.ReLU(True),
    )

    self.b2=nn.Sequential(
        BaseConv2d(in_channels,n3x3red,kernel_size=1),
        BaseConv2d(n3x3red,n3x3, kernel_size=5,padding=2),
    )

    self.b3=nn.Sequential(
        BaseConv2d(in_channels,n5x5,kernel_size=1),
        BaseConv2d(n5x5red,n5x5, kernel_size=5,padding=2),
    )

    self.b4=nn.Sequential(
        nn.MaxPool2d(3,stride=1,padding=1),
        BaseConv2d(in_channels,pool_proj,kernel_size=1),
    )

  def forward(self,x):
    y1=self.b1(x)
    y2=self.b2(x)
    y3=self.b3(x)
    y4=self.b4(x)
    return torch.cat([y1,y2,y3,y4],1)


class AuxilaryClassifier(nn.Module):
  def __init__(self,in_channels,num_classes,dropout=0.7):
    super(AuxilaryClassifier,self).__init__()
    self.pool=nn.AvgPool2d(5,stride=3)
    self.conv=BaseConv2d(in_channels,128,kernel_size=1)
    self.relu=nn.ReLU(True)
    self.flatten=nn.Flatten()
    self.fc1=nn.Linear(2048,1024)
    self.dropout=nn.Dropout(dropout)
    self.fc2=nn.Linear(1024,num_classes)

    def forward(self,x):
      x=self.pool(x)
      x=self.conv(x)
      x=self.flatten(x)
      x=self.fc1(x)
      x=self.relu(x)
      x=self.dropout(x)
      x=self.fc2(X)
      return x


class GoogLeNet(nn.Module):
    def __init__(self, use_aux=True):
        super(GoogLeNet, self).__init__()

        self.use_aux = use_aux
        ## block 1
        self.conv1 = BaseConv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.lrn1 = nn.LocalResponseNorm(5, alpha=0.0001, beta=0.75)
        self.maxpool1 = nn.MaxPool2d(3, stride=2, padding=1)

        ## block 2
        self.conv2 = BaseConv2d(64, 64, kernel_size=1)
        self.conv3 = BaseConv2d(64, 192, kernel_size=3, padding=1)
        self.lrn2 = nn.LocalResponseNorm(5, alpha=0.0001, beta=0.75)
        self.maxpool2 = nn.MaxPool2d(3, stride=2, padding=1)

        ## block 3
        self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(3, stride=2, padding=1)

        ## block 4
        self.inception4a = InceptionModule(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = InceptionModule(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = InceptionModule(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = InceptionModule(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = InceptionModule(528, 256, 160, 320, 32, 128, 128)
        self.maxpool4 = nn.MaxPool2d(3, stride=2, padding=1)

        ## block 5
        self.inception5a = InceptionModule(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = InceptionModule(832, 384, 192, 384, 48, 128, 128)

        ## auxiliary classifier
        if self.use_aux:
            self.aux1 = AuxiliaryClassifier(512, 1000)
            self.aux2 = AuxiliaryClassifier(528, 1000)

        ## block 6
        self.avgpool = nn.AvgPool2d(7, stride=1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, 1000)

    def forward(self, x):
        ## block 1
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.lrn1(x)

        ## block 2
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.lrn2(x)
        x = self.maxpool2(x)

        ## block 3
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)

        ## block 4
        x = self.inception4a(x)
        if self.use_aux:
            aux1 = self.aux1(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        if self.use_aux:
            aux2 = self.aux2(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)

        ## block 5
        x = self.inception5a(x)
        x = self.inception5b(x)

        ## block 6
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        if self.use_aux:
            return x, aux1, aux2
        else:
            return x